
# Daily FLEXPART footprint postprocessing

This notebook postprocesses daily FLEXPART `grid_time_*.nc` output where each file contains 24 hourly releases. The FLEXPART `pointspec` dimension is kept as the release dimension and is written as 24 `time` slices in the postprocessed footprint file.


In [ ]:

from __future__ import annotations

import calendar
import datetime as dt
import re
import sys
from pathlib import Path

import numpy as np
import xarray as xr

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "run_scripts").exists():
    REPO_ROOT = Path("/home/lwestern/work/flexpart_gfs")
sys.path.insert(0, str(REPO_ROOT / "run_scripts"))

from postprocess_footprint import (
    MOLAR_MASS_AIR_KG_PER_MOL,
    _build_time_attrs,
    _convert_to_m2s_per_mol,
    _find_spatial_dims,
    _pick_sensitivity_var,
    _set_netcdf_compression,
)

DEFAULT_FLEXPART_OUTPUT_DIR = Path("/net/fs06/d2/lwestern/flexpart_outs_daily")
POSTPROCESSED_ROOT = DEFAULT_FLEXPART_OUTPUT_DIR

FOOTPRINT_OUTHEIGHT_M = 100.0
SOURCE_LAYER_THICKNESS_M = 100.0
SPECIES = "inert"
MODEL = "FLEXPART"
MET_MODEL = "CFSv2"
OVERWRITE = False
COMPRESSION_LEVEL = 4


In [ ]:

def open_grid_dataset(path: str | Path) -> xr.Dataset:
    """Open FLEXPART NetCDF with numeric time coordinates preserved."""
    path = str(path)
    try:
        return xr.open_dataset(path, engine="netcdf4", decode_times=False)
    except Exception:
        return xr.open_dataset(path, decode_times=False)


def parse_run_dir_name(path: str | Path):
    """Return (domain, site, date) from DOMAIN_SITE_YYYYMMDD directories."""
    name = Path(path).name
    match = re.match(r"^(.+)_([^_]+)_(\d{8})$", name)
    if match is None:
        return None
    domain, site, yyyymmdd = match.groups()
    return domain, site, dt.datetime.strptime(yyyymmdd, "%Y%m%d").date()


def discover_daily_run_dirs(root: str | Path = DEFAULT_FLEXPART_OUTPUT_DIR) -> list[Path]:
    root = Path(root)
    return sorted(p for p in root.glob("*") if p.is_dir() and parse_run_dir_name(p) is not None)


def group_run_dirs_by_site_month(run_dirs: list[Path]):
    grouped = {}
    for run_dir in run_dirs:
        parsed = parse_run_dir_name(run_dir)
        if parsed is None:
            continue
        domain, site, date = parsed
        key = (domain, site, date.year, date.month)
        grouped.setdefault(key, []).append(run_dir)
    return {key: sorted(paths) for key, paths in grouped.items()}


def complete_month_groups(root: str | Path = DEFAULT_FLEXPART_OUTPUT_DIR):
    for key, run_dirs in sorted(group_run_dirs_by_site_month(discover_daily_run_dirs(root)).items()):
        domain, site, year, month = key
        expected = calendar.monthrange(year, month)[1]
        if len(run_dirs) == expected:
            yield key, run_dirs
        else:
            print(f"Skipping {domain}_{site} {year}-{month:02d}: {len(run_dirs)} days found, expected {expected}")


In [ ]:

def _as_float_seconds(values) -> np.ndarray:
    arr = np.asarray(values)
    if np.issubdtype(arr.dtype, np.timedelta64):
        return arr.astype("timedelta64[s]").astype(np.float64)
    if np.issubdtype(arr.dtype, np.datetime64):
        return arr.astype("datetime64[s]").astype(np.float64)
    return arr.astype(np.float64)


def release_time_values(ds: xr.Dataset, release_dim: str, nrelease: int) -> np.ndarray:
    """Return one numeric time coordinate value per hourly release."""
    for name in ["RELSTART", "RELEND"]:
        if name not in ds:
            continue
        da = ds[name]
        if release_dim in da.dims or "numpoint" in da.dims or da.size == nrelease:
            values = _as_float_seconds(da.values)
            return values[:nrelease]
    return np.arange(nrelease, dtype=np.float64)


def release_metadata_1d(ds: xr.Dataset, name: str, release_dim: str, nrelease: int, default=np.nan):
    """Pull a numpoint/pointspec release metadata variable onto output time."""
    if name not in ds:
        return np.full(nrelease, default, dtype=np.float32)
    da = ds[name]
    if release_dim in da.dims:
        values = np.asarray(da.values)[:nrelease]
    else:
        values = np.asarray(da.values).ravel()
        values = np.resize(values, nrelease)
    return values


def compute_daily_srr(ds: xr.Dataset, footprint_outheight_m: float = FOOTPRINT_OUTHEIGHT_M):
    """Integrate backward sensitivity for each hourly release in a daily grid file."""
    var_name = _pick_sensitivity_var(ds)
    da = ds[var_name]

    if "pointspec" in da.dims:
        release_dim = "pointspec"
    elif "numpoint" in da.dims:
        release_dim = "numpoint"
    else:
        release_dim = None

    selected_height = None
    integrated_height_levels = None
    if "height" in da.dims:
        hvals = np.asarray(ds["height"].values if "height" in ds else da["height"].values, dtype=float)
        hidx = int(np.argmin(np.abs(hvals - float(footprint_outheight_m))))
        selected_height = float(hvals[hidx])
        integrated_height_levels = hidx + 1
        da = da.isel(height=slice(0, hidx + 1))
        if not np.isclose(selected_height, footprint_outheight_m, atol=1e-6):
            print(f"Using nearest output height {selected_height:g} m for requested {footprint_outheight_m:g} m")

    reduce_dims = ["time", "height", "nageclass", "numspec"]
    reduce_dims = [dim for dim in reduce_dims if dim in da.dims]
    if reduce_dims:
        da = da.sum(dim=reduce_dims, skipna=True)

    native_units = ds[var_name].attrs.get("units", "")
    da, conv_factor = _convert_to_m2s_per_mol(da, native_units, SOURCE_LAYER_THICKNESS_M)

    if release_dim is None:
        da = da.expand_dims(time=np.array([0.0], dtype=np.float64))
        nrelease = 1
    else:
        nrelease = da.sizes[release_dim]
        da = da.rename({release_dim: "time"})
        da = da.assign_coords(time=release_time_values(ds, release_dim, nrelease))

    lat_dim, lon_dim = _find_spatial_dims(ds)
    rename_dims = {}
    if lat_dim != "latitude":
        rename_dims[lat_dim] = "latitude"
    if lon_dim != "longitude":
        rename_dims[lon_dim] = "longitude"
    if rename_dims:
        da = da.rename(rename_dims)

    da = da.transpose("time", "latitude", "longitude").astype(np.float32)
    da.attrs.update({
        "long_name": "source_receptor_relationship",
        "loss_lifetime_hrs": -9.0,
        "loss_lifetime_comment": "lifetime in hours; -9 corresponds to inert",
        "units": "m2 s mol-1",
        "source_variable": var_name,
        "description": "time-integrated hourly-release footprint from daily FLEXPART grid_time output",
        "conversion_from_native_units": str(native_units),
        "conversion_factor_applied": float(conv_factor),
        "source_layer_thickness_m": float(SOURCE_LAYER_THICKNESS_M),
        "molar_mass_air_kg_per_mol": float(MOLAR_MASS_AIR_KG_PER_MOL),
    })
    if selected_height is not None:
        da.attrs.update({
            "footprint_outheight_m": float(selected_height),
            "requested_footprint_outheight_m": float(footprint_outheight_m),
            "footprint_height_integration": "surface_to_outheight_inclusive",
            "footprint_height_levels_integrated": int(integrated_height_levels),
        })

    return da, var_name, release_dim or "time"


In [ ]:

def build_daily_footprint_dataset(grid_file: str | Path, domain: str, site: str) -> xr.Dataset:
    """Build one postprocessed dataset with 24 hourly release footprints."""
    grid_file = Path(grid_file)
    ds = open_grid_dataset(grid_file)
    try:
        srr, var_name, release_dim = compute_daily_srr(ds)
        ntime = srr.sizes["time"]

        out = xr.Dataset(coords={
            "time": srr["time"],
            "latitude": ds["latitude"],
            "longitude": ds["longitude"],
        })
        out["srr"] = srr

        out["time"].attrs.update(_build_time_attrs(ds))
        out["time"].attrs["comment"] = "hourly release-time stamps from RELSTART/RELEND in daily FLEXPART output"
        out["latitude"].attrs.update({"units": "degrees_north", "long_name": "latitude"})
        out["longitude"].attrs.update({"units": "degrees_east", "long_name": "longitude"})

        release_vars = {
            "release_lon": ("RELLNG1", "degrees_east", "Release longitude"),
            "release_lat": ("RELLAT1", "degrees_north", "Release latitude"),
            "release_height": ("RELZZ1", "m", "Release height above model ground"),
            "release_height_top": ("RELZZ2", "m", "Release height top above model ground"),
            "release_particles": ("RELPART", "1", "Number of release particles"),
        }
        for out_name, (src_name, units, long_name) in release_vars.items():
            values = release_metadata_1d(ds, src_name, "numpoint", ntime)
            dtype = np.int32 if src_name == "RELPART" else np.float32
            out[out_name] = xr.DataArray(
                values.astype(dtype),
                dims=("time",),
                coords={"time": out["time"]},
                attrs={"units": units, "long_name": long_name},
            )

        for name, units, long_name in [
            ("air_temperature", "K", "air temperature at release"),
            ("air_pressure", "hPa", "air pressure at release"),
            ("wind_speed", "m s-1", "wind speed at release"),
            ("wind_from_direction", "degree", "wind direction at release"),
            ("atmosphere_boundary_layer_thickness", "m", "atmospheric boundary layer thickness at release"),
        ]:
            out[name] = xr.DataArray(
                np.full(ntime, np.nan, dtype=np.float32),
                dims=("time",),
                coords={"time": out["time"]},
                attrs={"units": units, "long_name": long_name},
            )

        out.attrs.update({
            "title": "Derived FLEXPART daily hourly-release footprint products",
            "input_grid_file": str(grid_file.resolve()),
            "note": "Backward grid_time is source-receptor sensitivity; pointspec is written as hourly release time.",
            "lpdm_native_output_unit": "s",
            "species": SPECIES,
            "model": MODEL,
            "met_model": MET_MODEL,
            "output_folder": "output",
            "model_version": str(ds.attrs.get("source", "FLEXPART")),
            "domain": domain,
            "site": site,
            "author": "run_scripts_daily/test_postprocess.ipynb",
            "created": dt.datetime.utcnow().isoformat() + "Z",
            "source_sensitivity_variable": var_name,
            "release_dimension_in_input": release_dim,
        })
        return out
    finally:
        ds.close()


def output_name_for_run(run_dir: str | Path, grid_file: str | Path) -> str:
    domain, site, date = parse_run_dir_name(run_dir)
    ds = open_grid_dataset(grid_file)
    try:
        h = float(np.asarray(ds["RELZZ1"].values).ravel()[0]) if "RELZZ1" in ds else np.nan
    finally:
        ds.close()
    height_label = "unknown" if not np.isfinite(h) else (str(int(h)) if h.is_integer() else f"{h:.3f}".rstrip("0").rstrip("."))
    return f"{site}-{height_label}magl_FLEXPART_CFSv2_{domain}_{SPECIES}_{date:%Y%m%d}.nc"


def process_daily_run(run_dir: str | Path, output_root: str | Path = POSTPROCESSED_ROOT, overwrite: bool = OVERWRITE):
    run_dir = Path(run_dir)
    parsed = parse_run_dir_name(run_dir)
    if parsed is None:
        raise ValueError(f"Could not parse run directory name: {run_dir}")
    domain, site, _ = parsed

    grid_files = sorted((run_dir / "output").glob("grid_time_*.nc"))
    if not grid_files:
        print(f"No grid_time files found in {run_dir / 'output'}")
        return None
    if len(grid_files) > 1:
        print(f"Found {len(grid_files)} grid_time files; using {grid_files[-1].name}")
    grid_file = grid_files[-1]

    out_dir = Path(output_root) / run_dir.name / "output"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / output_name_for_run(run_dir, grid_file)
    if out_file.exists() and not overwrite:
        print(f"SKIP existing: {out_file}")
        return out_file

    out = build_daily_footprint_dataset(grid_file, domain=domain, site=site)
    encoding = _set_netcdf_compression(out, compression_level=COMPRESSION_LEVEL)
    out.to_netcdf(out_file, encoding=encoding)
    out.close()
    print(f"Wrote {out_file}")
    return out_file


In [ ]:

# Quick single-day check. This should produce one file with srr(time=24, latitude, longitude).
SAMPLE_RUN_DIR = DEFAULT_FLEXPART_OUTPUT_DIR / "WESTUSA_THD_20180101"
SAMPLE_GRID_FILE = SAMPLE_RUN_DIR / "output" / "grid_time_20180101230000.nc"

if SAMPLE_GRID_FILE.exists():
    sample = build_daily_footprint_dataset(SAMPLE_GRID_FILE, domain="WESTUSA", site="THD")
    print(sample)
    print("srr shape:", sample["srr"].shape)
    sample.close()
else:
    print(f"Sample grid file not found: {SAMPLE_GRID_FILE}")


In [ ]:

# Write the sample day when you are ready.
WRITE_SAMPLE = False

if WRITE_SAMPLE:
    process_daily_run(SAMPLE_RUN_DIR, overwrite=OVERWRITE)


In [ ]:

# Batch process complete months.
PROCESS_COMPLETE_MONTHS = False
MAX_MONTHS = None

if PROCESS_COMPLETE_MONTHS:
    processed = []
    for month_index, (key, run_dirs) in enumerate(complete_month_groups(DEFAULT_FLEXPART_OUTPUT_DIR), start=1):
        domain, site, year, month = key
        print(f"Processing {domain}_{site} {year}-{month:02d} ({len(run_dirs)} days)")
        for run_dir in run_dirs:
            out_file = process_daily_run(run_dir, overwrite=OVERWRITE)
            if out_file is not None:
                processed.append(out_file)
        if MAX_MONTHS is not None and month_index >= MAX_MONTHS:
            break
    print(f"Processed {len(processed)} daily footprint files")
